# Descarga diaria de la temperatura en los Aeropuertos

In [0]:
# ==============================
# METAR Daily Ingestion - Databricks
# ==============================

import json
from datetime import datetime, timedelta, timezone, time
from pathlib import Path
import requests

# -------- CONFIG --------
BASE_URL = "https://aviationweather.gov/api/data/metar"
AIRPORTS = ["SBGR", "SBCT", "SBPA", "SBFL"]
VOLUME_BASE = Path("/Volumes/weather/raw/noaa_volume")
DAILY_JSON_DIR = VOLUME_BASE / "json" / "daily"
DAILY_JSON_DIR.mkdir(parents=True, exist_ok=True)
# ------------------------

def yesterday_utc():
    return (datetime.now(timezone.utc) - timedelta(days=1)).date()

def hours_needed_for_yesterday():
    now = datetime.now(timezone.utc)
    start_yesterday = datetime.combine(yesterday_utc(), time.min, tzinfo=timezone.utc)
    delta_hours = int((now - start_yesterday).total_seconds() // 3600) + 1
    return max(24, min(delta_hours, 72))

def fetch_metars(airports, hours):
    params = {
        "ids": ",".join(airports),
        "hours": hours,
        "format": "json",
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if not isinstance(data, list):
        raise RuntimeError("Formato inesperado de respuesta METAR")
    return data

def filter_by_date(records, target_date):
    filtered = []
    for r in records:
        obs = r.get("obsTime")
        dt = None
        if isinstance(obs, (int, float)):
            dt = datetime.fromtimestamp(int(obs), tz=timezone.utc)
        elif isinstance(obs, str):
            try:
                dt = datetime.fromisoformat(obs.replace("Z", "+00:00"))
            except Exception:
                continue
        if dt and dt.date() == target_date:
            filtered.append(r)
    return filtered

def record_signature(r):
    return (
        r.get("icaoId") or r.get("stationId"),
        r.get("obsTime"),
        r.get("rawText"),
    )

def records_equal(a, b):
    return sorted(record_signature(x) for x in a) == \
           sorted(record_signature(x) for x in b)

# -------- EXECUTION --------

target_date = yesterday_utc()
hours = hours_needed_for_yesterday()

print(f"Procesando METAR para fecha: {target_date} | hours={hours}")

records = fetch_metars(AIRPORTS, hours)
records = filter_by_date(records, target_date)

output_file = DAILY_JSON_DIR / f"METAR_{target_date:%Y_%m_%d}.json"

if output_file.exists():
    with output_file.open("r", encoding="utf-8") as f:
        previous_data = json.load(f)
else:
    previous_data = []

write_file = False

if not previous_data:
    write_file = True
elif records_equal(previous_data, records):
    print("✓ Archivo ya actualizado. No se reemplaza.")
elif len(records) > len(previous_data):
    print(f"✓ Reemplazando archivo: nuevos registros {len(records)} > {len(previous_data)}")
    write_file = True
else:
    print("✓ Archivo existente es más completo. Se conserva.")

if write_file:
    with output_file.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)
    print(f"✓ Guardado: {output_file}")

print(f"Total registros día anterior: {len(records)}")